# Catalogue ingestion walkthrough

This notebook shows, step by step, how a CSV row is validated and turned into the values that will be upserted into PostgreSQL by `validate_and_upsert_catalogue`.

## 1. Imports & helpers
We reuse the same helpers as the Airflow DAG so the behavior matches production.

In [ ]:
from pathlib import Path

import pandas as pd

from scripts.load_to_postgres import validate_flow
from contracts.catalogue_contract import get_catalogue_contract
from contracts.catalogue_schema import get_catalogue_storage_columns
from scripts.db_schema_manager import ensure_products_columns

CATALOGUE_DB_COLUMNS = get_catalogue_storage_columns()
CATALOGUE_DB_COLUMNS

## 2. Point to the CSV you want to inspect
Update `csv_path` if you want to try another file (for example a file dropped into `data/inbox/catalogue/`).

In [ ]:
csv_path = Path("tests/data/catalogue_test_v2.csv")  # 👈 change me
df = pd.read_csv(csv_path)
df

## 3. Validate rows using the matching contract per schema version
This mirrors what `validate_and_upsert_catalogue` does: it runs `validate_flow`, then gives you the valid vs rejected DataFrames.

In [ ]:
valid_df, rejected_df = validate_flow(str(csv_path), get_catalogue_contract("2.0"))
display(valid_df)
display(rejected_df)

## 4. See the exact tuples that will be sent to PostgreSQL
We align each validated payload on the superset of columns (across every schema version). Missing fields become `None`, which translates to `NULL` in the database.

In [ ]:
db_values = [
    tuple(row.get(col) for col in CATALOGUE_DB_COLUMNS)
    for _, row in valid_df.iterrows()
]
CATALOGUE_DB_COLUMNS, db_values

## 5. (Optional) Push to your local PostgreSQL
Set `RUN_DB_LOAD = True` once your Docker stack is up (`docker-compose up -d`). This cell mirrors the DAG behavior: it ensures the table has every required column and performs the UPSERT using psycopg2.

In [ ]:
import os
import psycopg2
from psycopg2.extras import execute_values

POSTGRES_DSN = {
    "host": os.getenv("POSTGRES_HOST", "localhost"),
    "port": int(os.getenv("POSTGRES_PORT", 5432)),
    "dbname": os.getenv("POSTGRES_DB", "logistore"),
    "user": os.getenv("POSTGRES_USER", "logistore"),
    "password": os.getenv("POSTGRES_PASSWORD", "logistore"),
}

RUN_DB_LOAD = False  # flip to True to execute the UPSERT

if RUN_DB_LOAD and db_values:
    insert_columns = ", ".join(CATALOGUE_DB_COLUMNS)
    set_clause = ", ".join(
        f"{col} = EXCLUDED.{col}" for col in CATALOGUE_DB_COLUMNS if col != "sku"
    )
    upsert_query = f"""
        INSERT INTO products ({insert_columns})
        VALUES %s
        ON CONFLICT (sku) DO UPDATE SET {set_clause}
    """
    with psycopg2.connect(**POSTGRES_DSN) as conn:
        ensure_products_columns(conn)
        with conn.cursor() as cur:
            execute_values(cur, upsert_query, db_values)
        conn.commit()
    print(f"Upserted {len(db_values)} rows into products table")
else:
    print("Set RUN_DB_LOAD = True to push these rows into PostgreSQL.")